In [1]:
# ==========================================
#   IMPORT LIBRARIES & SETUP
# ==========================================
%load_ext autoreload
%autoreload 2

from metrics import evaluate
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np
from scipy.sparse import csr_matrix
import implicit

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## DATA LOADING

In [2]:
articles=pd.read_parquet('data/articles_processed.parquet')
customers=pd.read_parquet('data/customers_processed.parquet')
transactions=pd.read_parquet('data/transactions_processed.parquet')


In [3]:
#=================================================================================
#                        MAPPING
#=================================================================================

unique_customers = customers['customer_id'].unique()
unique_articles = articles['article_id'].unique()

# Mappings
customer2idx = {c: i for i, c in enumerate(unique_customers)}
article2idx = {a: i for i, a in enumerate(unique_articles)}

# Reverse Mappings 
idx2customer = {i: c for c, i in customer2idx.items()}
idx2article = {int(i): a for a, i in article2idx.items()}

# Mapping transactions-ի վրա
transactions['user_idx'] = transactions['customer_id'].map(customer2idx)
transactions['item_idx'] = transactions['article_id'].map(article2idx)

In [4]:
#================================================================================
#                  TRAIN-VALIDATION SPLIT
#================================================================================
last_date = transactions['t_dat'].max()
val_start = last_date - pd.Timedelta(days=6)
train = transactions[transactions['t_dat'] < val_start]
val = transactions[transactions['t_dat'] >= val_start]

print(transactions['t_dat'].dtype)
print(len(train), len(val))
val_customers = set(val['user_idx'].unique())
train_customers = set(train['user_idx'].unique())
print(len(val_customers), len(val_customers - train_customers))

datetime64[ns]
31548013 240311
68984 5572


* val-ում 68 984  հաճախորդ կա, որոնցից 5 572-ը train-ում չկան

In [5]:
#=================================================================================
#                              CSR Matrix
#=================================================================================

train_user_item = train.groupby(['user_idx', 'item_idx']).size().reset_index(name='weight')

rows = train_user_item['user_idx'].values
cols = train_user_item['item_idx'].values
weights = train_user_item['weight'].values.astype(np.float32)

n_users = len(customer2idx)
n_items = len(article2idx)

#  User-Item Matrix
user_item_matrix = csr_matrix((weights, (rows, cols)), shape=(n_users, n_items))

# Item-User Matrix
item_user_matrix = user_item_matrix.T.tocsr()

## Collaborative Filtering

In [7]:

recent_popular = train[train['t_dat'] >= train['t_dat'].max() - pd.Timedelta(days=6)]
fallback_top12 = recent_popular.groupby('article_id').size().nlargest(12).index.tolist()

def predict_implicit_model(model, val_df, k=12):
    
    actuals_dict = val_df.groupby('customer_id')['article_id'].unique().apply(set).to_dict()
    val_customers = list(actuals_dict.keys())

    user_idxs = np.array([customer2idx[c_id] for c_id in val_customers])

    has_history = np.asarray(user_item_matrix[user_idxs].getnnz(axis=1)) > 0

    ids, scores = model.recommend(
        user_idxs,
        user_item_matrix[user_idxs],
        N=k,
        filter_already_liked_items=False
    )

    predictions = {}
    for i, c_id in enumerate(val_customers):
        if has_history[i]:
            recs = [idx2article[int(idx)] for idx in ids[i] if idx >= 0]
        else:
            recs = []  # cold customer: the model has nothing to say,
    # fill the missing slots with popular items (no duplicates)
        fill = [p for p in fallback_top12 if p not in recs]
        predictions[c_id] = (recs + fill)[:k]

    return predictions, actuals_dict

In [9]:
#=================================================================================
#                   Item-Item Cosine Similarity
#=================================================================================

cosine_model = implicit.nearest_neighbours.CosineRecommender(K=20)
cosine_model.fit(user_item_matrix)
print("Cosine Similarity model successfully fitted!")


C:\Users\user\AppData\Roaming\Python\Python314\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.15923023223876953 seconds
  warnings.warn(
100%|██████████| 105542/105542 [00:08<00:00, 12460.40it/s]


Cosine Similarity model successfully fitted!


In [10]:
cosine_preds, actuals_dict = predict_implicit_model(cosine_model, val, k=12)

In [ ]:
 3. Metrics calculation
total_catalog_items = len(unique_articles)
cosine_results = evaluate_baseline(cosine_preds, actuals_dict, total_catalog_items, k=12)

print("--- Cosine Similarity Results ---")
for metric_name, value in cosine_results.items():
    print(f"{metric_name}: {value:.5f}")